# Context Engine Testing Notebook

This notebook tests the ContextEngine for ingestion and retrieval using sample files.

In [9]:
from context_engine import ContextEngine, ContextEngineConfig
# from context_engine.schema import Doc, ParsedDoc
import os

## Setup

Configure the ContextEngine with default settings using HashEmbedder for local testing.

In [10]:
# Define the data directory
DATA_DIR = "/Users/nnandagopal/Desktop/personal_projects/RAG/context_engine/data"
SAMPLE_PDF = os.path.join(DATA_DIR, "coffee_processing.pdf")

print(f"Data directory: {DATA_DIR}")
print(f"Sample PDF: {SAMPLE_PDF}")
print(f"PDF exists: {os.path.exists(SAMPLE_PDF)}")

Data directory: /Users/nnandagopal/Desktop/personal_projects/RAG/context_engine/data
Sample PDF: /Users/nnandagopal/Desktop/personal_projects/RAG/context_engine/data/coffee_processing.pdf
PDF exists: True


## Initialize ContextEngine

Create a ContextEngine instance with ChromaDB store and HashEmbedder for local testing.

In [11]:
config = ContextEngineConfig(
    loader="auto",
    chunker="recursive",
    store="chroma",
    store_kwargs={
        "collection_name": "coffee_test",
        "persist_directory": ".rag_chroma_test",
    },
    embedder="hash",  # local deterministic embeddings for testing
    chunker_kwargs={"chunk_size": 500, "chunk_overlap": 80},
)

engine = ContextEngine(config)
print("ContextEngine initialized successfully")

ContextEngine initialized successfully


## Test 1: Parse Document

Test parsing a PDF document without chunking.

In [12]:
parsed_doc = engine.parse(SAMPLE_PDF)
print(f"Parsed document type: {type(parsed_doc)}")
print(f"Number of texts: {len(parsed_doc.texts)}")
print(f"Number of images: {len(parsed_doc.images)}")
print(f"Number of tables: {len(parsed_doc.tables)}")
print(f"\nFirst 200 characters of first text:")
print(parsed_doc.texts[0][:200] if parsed_doc.texts else "No texts found")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 360/360 [00:00<00:00, 12206.74it/s]
[INFO] 2026-05-17 16:46:59,721 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-05-17 16:46:59,735 [RapidOCR] download_file.py:60: File exists and is valid: /Users/nnandagopal/Desktop/personal_projects/RAG/.venv/lib/python3.12/site-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.onnx
[INFO] 2026-05-17 16:46:59,736 [RapidOCR] main.py:57: Using /Users/nnandagopal/Desktop/personal_projects/RAG/.venv/lib/python3.12/site-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.onnx
[INFO] 2026-05-17 16:46:59,758 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-05-17 16:46:59,762 [RapidOCR] download_file.py:60: File exists and is valid: /Users/nnandagopal/Desktop/personal_projects/RAG/.venv/lib/python3.12/site-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-05-17 16:46:59,763 [RapidOCR] main.py:57: Using /Use

Parsed document type: <class 'context_engine.loaders.schema.ParsedDoc'>
Number of texts: 1
Number of images: 0
Number of tables: 0

First 200 characters of first text:
## Coffee Processing Methods

A Comprehensive Guide to Post-Harvest Coffee Processing

Coffee processing began in Ethiopia over 1,000 years ago.

## Introduction to Coffee Processing

Coffee processin


## Test 2: Chunk Document

Test chunking the parsed document.

In [13]:
chunks = engine.chunk(parsed_doc)
print(f"Number of chunks: {len(chunks)}")
print(f"\nFirst chunk (first 300 characters):")
print(chunks[0].page_content[:300] if chunks else "No chunks found")
print(f"\nFirst chunk metadata:")
print(chunks[0].metadata if chunks else "No metadata")

Number of chunks: 22

First chunk (first 300 characters):
## Coffee Processing Methods A Comprehensive Guide to Post-Harvest Coffee Processing Coffee processing began in Ethiopia over 1,000 years ago. ## Introduction to Coffee Processing

First chunk metadata:
{'chunk_id': 0, 'source': 'document', 'chunker': 'recursive'}


## Test 3: Run Full Pipeline

Test the complete parse + chunk pipeline in one call.

In [14]:
docs = engine.run(SAMPLE_PDF)
print(f"Number of documents from run(): {len(docs)}")
print(f"\nSample document (first 300 characters):")
print(docs[0].page_content[:300] if docs else "No documents found")

Loading weights: 100%|██████████| 360/360 [00:00<00:00, 26313.99it/s]
[INFO] 2026-05-17 16:47:17,068 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-05-17 16:47:17,071 [RapidOCR] download_file.py:60: File exists and is valid: /Users/nnandagopal/Desktop/personal_projects/RAG/.venv/lib/python3.12/site-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.onnx
[INFO] 2026-05-17 16:47:17,071 [RapidOCR] main.py:57: Using /Users/nnandagopal/Desktop/personal_projects/RAG/.venv/lib/python3.12/site-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.onnx
[INFO] 2026-05-17 16:47:17,088 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-05-17 16:47:17,089 [RapidOCR] download_file.py:60: File exists and is valid: /Users/nnandagopal/Desktop/personal_projects/RAG/.venv/lib/python3.12/site-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-05-17 16:47:17,089 [RapidOCR] main.py:57: Using /Users/nnandagopal/Desktop/personal_projects/RAG/.venv/lib/python3.12

Number of documents from run(): 22

Sample document (first 300 characters):
## Coffee Processing Methods A Comprehensive Guide to Post-Harvest Coffee Processing Coffee processing began in Ethiopia over 1,000 years ago. ## Introduction to Coffee Processing


## Test 4: Ingest Documents

Ingest documents into the vector store.

In [15]:
# Clear any existing data first
engine.clear()
print("Cleared existing data from store")

# Ingest the document
doc_ids = engine.ingest(SAMPLE_PDF)
print(f"Ingested {len(doc_ids)} documents")
print(f"Document IDs (first 5): {doc_ids[:5]}")
print(f"Total documents in store: {engine.count()}")

Cleared existing data from store


Loading weights: 100%|██████████| 360/360 [00:00<00:00, 20012.05it/s]
[INFO] 2026-05-17 16:47:25,146 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-05-17 16:47:25,148 [RapidOCR] download_file.py:60: File exists and is valid: /Users/nnandagopal/Desktop/personal_projects/RAG/.venv/lib/python3.12/site-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.onnx
[INFO] 2026-05-17 16:47:25,148 [RapidOCR] main.py:57: Using /Users/nnandagopal/Desktop/personal_projects/RAG/.venv/lib/python3.12/site-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.onnx
[INFO] 2026-05-17 16:47:25,165 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-05-17 16:47:25,166 [RapidOCR] download_file.py:60: File exists and is valid: /Users/nnandagopal/Desktop/personal_projects/RAG/.venv/lib/python3.12/site-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-05-17 16:47:25,166 [RapidOCR] main.py:57: Using /Users/nnandagopal/Desktop/personal_projects/RAG/.venv/lib/python3.12

Ingested 22 documents
Document IDs (first 5): ['document:0', 'document:1', 'document:2', 'document:3', 'document:4']
Total documents in store: 22


## Test 5: Retrieve Documents

Test retrieval with different queries.

In [16]:
# Test retrieval with a coffee processing query
query = "How is washed coffee processed?"
results = engine.retrieve(query, k=3)

print(f"Query: {query}")
print(f"Number of results: {len(results)}")
print(f"\nResult 1 (first 400 characters):")
print(results[0].page_content[:400] if results else "No results")
print(f"\nResult 1 metadata:")
print(results[0].metadata if results else "No metadata")

Query: How is washed coffee processed?
Number of results: 3

Result 1 (first 400 characters):
## Coffee Processing Methods A Comprehensive Guide to Post-Harvest Coffee Processing Coffee processing began in Ethiopia over 1,000 years ago. ## Introduction to Coffee Processing

Result 1 metadata:
{'chunk_id': 0, 'source': 'document', 'chunker': 'recursive', 'distance': 0.763060450553894}


In [17]:
# Test with another query
query2 = "What is natural coffee processing?"
results2 = engine.retrieve(query2, k=2)

print(f"Query: {query2}")
print(f"Number of results: {len(results2)}")
print(f"\nResult 1 (first 400 characters):")
print(results2[0].page_content[:400] if results2 else "No results")

Query: What is natural coffee processing?
Number of results: 2

Result 1 (first 400 characters):
## Coffee Processing Methods A Comprehensive Guide to Post-Harvest Coffee Processing Coffee processing began in Ethiopia over 1,000 years ago. ## Introduction to Coffee Processing


## Test 6: Search Method

Test the simplified search method.

In [18]:
search_results = engine.search("coffee drying methods", k=3)
print(f"Search results count: {len(search_results)}")
print(f"\nFirst result (first 300 characters):")
print(search_results[0].page_content[:300] if search_results else "No results")

Search results count: 3

First result (first 300 characters):
## Coffee Processing Methods A Comprehensive Guide to Post-Harvest Coffee Processing Coffee processing began in Ethiopia over 1,000 years ago. ## Introduction to Coffee Processing


## Test 7: Different Chunker

Test with a different chunker (markdown chunker).

In [19]:
# Clear store for new test
engine.clear()

# Ingest with markdown chunker
doc_ids_md = engine.ingest(
    SAMPLE_PDF,
    chunker="markdown",
    chunker_kwargs={"chunk_size": 1000, "chunk_overlap": 150}
)
print(f"Ingested {len(doc_ids_md)} documents with markdown chunker")
print(f"Total documents in store: {engine.count()}")

# Retrieve with markdown chunked data
results_md = engine.retrieve("coffee processing methods", k=2)
print(f"\nRetrieved {len(results_md)} results")
print(f"\nFirst result (first 400 characters):")
print(results_md[0].page_content[:400] if results_md else "No results")

Loading weights: 100%|██████████| 360/360 [00:00<00:00, 15552.70it/s]
[INFO] 2026-05-17 16:47:42,740 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-05-17 16:47:42,742 [RapidOCR] download_file.py:60: File exists and is valid: /Users/nnandagopal/Desktop/personal_projects/RAG/.venv/lib/python3.12/site-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.onnx
[INFO] 2026-05-17 16:47:42,742 [RapidOCR] main.py:57: Using /Users/nnandagopal/Desktop/personal_projects/RAG/.venv/lib/python3.12/site-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.onnx
[INFO] 2026-05-17 16:47:42,757 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-05-17 16:47:42,759 [RapidOCR] download_file.py:60: File exists and is valid: /Users/nnandagopal/Desktop/personal_projects/RAG/.venv/lib/python3.12/site-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-05-17 16:47:42,759 [RapidOCR] main.py:57: Using /Users/nnandagopal/Desktop/personal_projects/RAG/.venv/lib/python3.12

Ingested 10 documents with markdown chunker
Total documents in store: 10

Retrieved 2 results

First result (first 400 characters):
## Coffee Processing Methods  
A Comprehensive Guide to Post-Harvest Coffee Processing  
Coffee processing began in Ethiopia over 1,000 years ago.


## Test 8: Tools Interface

Test the tools interface for agent integration.

In [20]:
tools = engine.tools()
print(f"Available tools: {list(tools.keys())}")

# Test using tools directly
engine.clear()

# Use ingest tool
ingested_ids = tools["context_ingest"](
    source=SAMPLE_PDF,
    chunker="recursive",
    chunker_kwargs={"chunk_size": 500, "chunk_overlap": 80}
)
print(f"Ingested via tools: {len(ingested_ids)} documents")

# Use retrieve tool
retrieved = tools["context_retrieve"](
    query="washed coffee processing",
    k=2
)
print(f"Retrieved via tools: {len(retrieved)} documents")
print(f"\nFirst result (first 300 characters):")
print(retrieved[0].page_content[:300] if retrieved else "No results")

Available tools: ['context_ingest', 'context_retrieve', 'context_parse', 'context_chunk']


Loading weights: 100%|██████████| 360/360 [00:00<00:00, 27231.32it/s]
[INFO] 2026-05-17 16:47:49,476 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-05-17 16:47:49,479 [RapidOCR] download_file.py:60: File exists and is valid: /Users/nnandagopal/Desktop/personal_projects/RAG/.venv/lib/python3.12/site-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.onnx
[INFO] 2026-05-17 16:47:49,480 [RapidOCR] main.py:57: Using /Users/nnandagopal/Desktop/personal_projects/RAG/.venv/lib/python3.12/site-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.onnx
[INFO] 2026-05-17 16:47:49,496 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-05-17 16:47:49,497 [RapidOCR] download_file.py:60: File exists and is valid: /Users/nnandagopal/Desktop/personal_projects/RAG/.venv/lib/python3.12/site-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-05-17 16:47:49,497 [RapidOCR] main.py:57: Using /Users/nnandagopal/Desktop/personal_projects/RAG/.venv/lib/python3.12

Ingested via tools: 22 documents
Retrieved via tools: 2 documents

First result (first 300 characters):
## Coffee Processing Methods A Comprehensive Guide to Post-Harvest Coffee Processing Coffee processing began in Ethiopia over 1,000 years ago. ## Introduction to Coffee Processing


## Test 9: Test with Different Sample File

Test with the SampleContract.pdf file.

In [21]:
CONTRACT_PDF = os.path.join(DATA_DIR, "SampleContract.pdf")
print(f"Contract PDF exists: {os.path.exists(CONTRACT_PDF)}")

if os.path.exists(CONTRACT_PDF):
    # Clear store
    engine.clear()
    
    # Ingest contract
    contract_ids = engine.ingest(
        CONTRACT_PDF,
        chunker="markdown",
        chunker_kwargs={"chunk_size": 800, "chunk_overlap": 100, "source": "contract"}
    )
    print(f"Ingested {len(contract_ids)} contract documents")
    
    # Retrieve from contract
    contract_results = engine.retrieve(
        "termination",
        k=3,
        where={"source": "contract"}
    )
    print(f"\nRetrieved {len(contract_results)} contract results")
    print(f"\nFirst result (first 400 characters):")
    print(contract_results[0].page_content[:400] if contract_results else "No results")

Contract PDF exists: True


Loading weights: 100%|██████████| 360/360 [00:00<00:00, 28046.17it/s]
[INFO] 2026-05-17 16:47:55,709 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-05-17 16:47:55,712 [RapidOCR] download_file.py:60: File exists and is valid: /Users/nnandagopal/Desktop/personal_projects/RAG/.venv/lib/python3.12/site-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.onnx
[INFO] 2026-05-17 16:47:55,712 [RapidOCR] main.py:57: Using /Users/nnandagopal/Desktop/personal_projects/RAG/.venv/lib/python3.12/site-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.onnx
[INFO] 2026-05-17 16:47:55,727 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-05-17 16:47:55,728 [RapidOCR] download_file.py:60: File exists and is valid: /Users/nnandagopal/Desktop/personal_projects/RAG/.venv/lib/python3.12/site-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-05-17 16:47:55,729 [RapidOCR] main.py:57: Using /Users/nnandagopal/Desktop/personal_projects/RAG/.venv/lib/python3.12

Ingested 59 contract documents

Retrieved 3 contract results

First result (first 400 characters):
CONSULTANT shall include in any subcontract with a third party for work under this Agreement terms that preserve the rights, interests, and obligations created by this Section, and that identify the COMMISSION as a third-party beneficiary of those provisions.  
The CONSULTANT shall not utilize the work produced under this Agreement for any profit-making venture, or sell or grant rights to a third 


## Test 10: Upsert vs Add

Test the difference between upsert and add operations.

In [22]:
engine.clear()

# First ingestion with upsert=True (default)
ids1 = engine.ingest(SAMPLE_PDF, upsert=True)
print(f"First ingestion (upsert): {len(ids1)} documents")
print(f"Total in store: {engine.count()}")

# Second ingestion with upsert=True (should replace)
ids2 = engine.ingest(SAMPLE_PDF, upsert=True)
print(f"\nSecond ingestion (upsert): {len(ids2)} documents")
print(f"Total in store: {engine.count()}")

# Clear and test with upsert=False (add)
engine.clear()
ids3 = engine.ingest(SAMPLE_PDF, upsert=False)
print(f"\nThird ingestion (add): {len(ids3)} documents")
print(f"Total in store: {engine.count()}")

# Add again with upsert=False (should duplicate)
ids4 = engine.ingest(SAMPLE_PDF, upsert=False)
print(f"\nFourth ingestion (add): {len(ids4)} documents")
print(f"Total in store: {engine.count()}")

Loading weights: 100%|██████████| 360/360 [00:00<00:00, 24270.25it/s]
[INFO] 2026-05-17 16:48:05,470 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-05-17 16:48:05,473 [RapidOCR] download_file.py:60: File exists and is valid: /Users/nnandagopal/Desktop/personal_projects/RAG/.venv/lib/python3.12/site-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.onnx
[INFO] 2026-05-17 16:48:05,473 [RapidOCR] main.py:57: Using /Users/nnandagopal/Desktop/personal_projects/RAG/.venv/lib/python3.12/site-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.onnx
[INFO] 2026-05-17 16:48:05,491 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-05-17 16:48:05,494 [RapidOCR] download_file.py:60: File exists and is valid: /Users/nnandagopal/Desktop/personal_projects/RAG/.venv/lib/python3.12/site-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-05-17 16:48:05,494 [RapidOCR] main.py:57: Using /Users/nnandagopal/Desktop/personal_projects/RAG/.venv/lib/python3.12

First ingestion (upsert): 22 documents
Total in store: 22


Loading weights: 100%|██████████| 360/360 [00:00<00:00, 28357.99it/s]
[INFO] 2026-05-17 16:48:09,212 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-05-17 16:48:09,215 [RapidOCR] download_file.py:60: File exists and is valid: /Users/nnandagopal/Desktop/personal_projects/RAG/.venv/lib/python3.12/site-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.onnx
[INFO] 2026-05-17 16:48:09,215 [RapidOCR] main.py:57: Using /Users/nnandagopal/Desktop/personal_projects/RAG/.venv/lib/python3.12/site-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.onnx
[INFO] 2026-05-17 16:48:09,231 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-05-17 16:48:09,232 [RapidOCR] download_file.py:60: File exists and is valid: /Users/nnandagopal/Desktop/personal_projects/RAG/.venv/lib/python3.12/site-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-05-17 16:48:09,232 [RapidOCR] main.py:57: Using /Users/nnandagopal/Desktop/personal_projects/RAG/.venv/lib/python3.12


Second ingestion (upsert): 22 documents
Total in store: 22


Loading weights: 100%|██████████| 360/360 [00:00<00:00, 24325.77it/s]
[INFO] 2026-05-17 16:48:13,311 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-05-17 16:48:13,313 [RapidOCR] download_file.py:60: File exists and is valid: /Users/nnandagopal/Desktop/personal_projects/RAG/.venv/lib/python3.12/site-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.onnx
[INFO] 2026-05-17 16:48:13,313 [RapidOCR] main.py:57: Using /Users/nnandagopal/Desktop/personal_projects/RAG/.venv/lib/python3.12/site-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.onnx
[INFO] 2026-05-17 16:48:13,331 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-05-17 16:48:13,332 [RapidOCR] download_file.py:60: File exists and is valid: /Users/nnandagopal/Desktop/personal_projects/RAG/.venv/lib/python3.12/site-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-05-17 16:48:13,333 [RapidOCR] main.py:57: Using /Users/nnandagopal/Desktop/personal_projects/RAG/.venv/lib/python3.12


Third ingestion (add): 22 documents
Total in store: 22


Loading weights: 100%|██████████| 360/360 [00:00<00:00, 30098.46it/s]
[INFO] 2026-05-17 16:48:17,270 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-05-17 16:48:17,273 [RapidOCR] download_file.py:60: File exists and is valid: /Users/nnandagopal/Desktop/personal_projects/RAG/.venv/lib/python3.12/site-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.onnx
[INFO] 2026-05-17 16:48:17,273 [RapidOCR] main.py:57: Using /Users/nnandagopal/Desktop/personal_projects/RAG/.venv/lib/python3.12/site-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.onnx
[INFO] 2026-05-17 16:48:17,289 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-05-17 16:48:17,290 [RapidOCR] download_file.py:60: File exists and is valid: /Users/nnandagopal/Desktop/personal_projects/RAG/.venv/lib/python3.12/site-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-05-17 16:48:17,290 [RapidOCR] main.py:57: Using /Users/nnandagopal/Desktop/personal_projects/RAG/.venv/lib/python3.12


Fourth ingestion (add): 22 documents
Total in store: 22


## Cleanup

Clear the vector store after testing.

In [23]:
engine.clear()
print(f"Store cleared. Final count: {engine.count()}")
print("\nAll tests completed successfully!")

Store cleared. Final count: 0

All tests completed successfully!
